In [1]:
import csv
import urllib.request, json
import os
import sys

# Get candidates

## PhaSePro

In [2]:
def get_phasepro():
    def json_parse(data):
        for key, value in data.items():
            prekey = key.split("-")[0]
            with open("phasepro_screened.txt", "at", encoding="utf-8") as out:
                out.write(prekey + "\n")

    with open("phasepro.json", "rt", encoding="utf-8") as json_file:
        data = json.load(json_file)
        json_parse(data)

## PhaSepDB

In [3]:
def get_phasepdb():
    uniprots = []
    with open("phasepdb_summary_database_2025-09-03.csv", "rt", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if not row:
                continue
            uniprots.append(row[0].strip())
    uniprots_filtered = list(set(uniprots))
    for protein in uniprots_filtered:
        preprotein = protein.split("-")[0]
        with open("phasepdb_screened.txt", "at", encoding="utf-8") as out:
            out.write(preprotein + "\n")

## LLPSDB

In [4]:
def get_llpsdb():
    llpsdb_ids = []
    with open("llpsdb_protein_unambiguous.csv", "rt", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if len(row) < 6:
                continue
            # Natural proteins, one protein component without repeats, PTMs or mutations
            if row[2] == "N" and row[5].strip() != "":
                llpsdb_ids.append(row[5].strip())
    for uniprot in llpsdb_ids:
        preuniprot = uniprot.split("-")[0]
        with open("llpsdb_screened.txt", "at") as out:
            out.write(preuniprot + "\n")

## DrLLPS

In [5]:
def drllps_first_screen(
    in_path="DrLLPS.txt",
    out_path="DrLLPS_first_screen.txt",
    ref_threshold=10,
):
    with open(in_path, "rt", encoding="utf-8") as f:
        lines = f.readlines()
    if not lines:
        return []
    header = lines[0].rstrip("\n")
    rows = []
    for line in lines[1:]:
        line = line.rstrip("\n")
        if not line:
            continue
        parts = line.split("\t", 8)
        if len(parts) < 9:
            continue
        drllps_id, uniprot, gene, ensembl, species, condensate, llps_type, references, seq = parts
        if not uniprot.strip():
            continue
        if llps_type.strip() == "Regulator":
            continue
        ref_tokens = [
            t.strip() for t in references.split(", ") if t.strip()
        ]
        rows.append((line, ref_tokens, llps_type.strip()))

    ref_counts = {}
    for _, ref_tokens, lt in rows:
        if lt != "Client":
            continue
        for r in set(ref_tokens):
            ref_counts[r] = ref_counts.get(r, 0) + 1
    bad_refs = {r for r, c in ref_counts.items() if c > ref_threshold}

    kept_lines = [header]
    for line, ref_tokens, lt in rows:
        if lt == "Client":
            if ref_tokens and all(r in bad_refs for r in ref_tokens):
                continue
        kept_lines.append(line)

    with open(out_path, "wt", encoding="utf-8") as out:
        out.write("\n".join(kept_lines))
        if kept_lines:
            out.write("\n")


def _drllps_detail_pmids(pmid_field):
    if not pmid_field:
        return []
    out = []
    for piece in pmid_field.replace(";", ",").split(","):
        t = piece.strip()
        if t.isdigit():
            out.append(t)
    return out


def drllps_second_screen(
    in_path="DrLLPS_first-screened_detail.csv",
    out_path="DrLLPS_screened.txt",
    pmid_threshold=10,
):
    with open(in_path, "rt", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))

    pmid_counts = {}
    for row in rows:
        if (row.get("Type") or "").strip() != "Client":
            continue
        for p in set(_drllps_detail_pmids(row.get("PMIDs") or "")):
            pmid_counts[p] = pmid_counts.get(p, 0) + 1
    bad_pmids = {p for p, c in pmid_counts.items() if c > pmid_threshold}

    uniprots = set()
    for row in rows:
        if (row.get("Type") or "").strip() == "Client":
            row_pmids = set(_drllps_detail_pmids(row.get("PMIDs") or ""))
            if row_pmids & bad_pmids:
                continue
        condensate = (row.get("Condensate") or "").strip()
        desc = (row.get("Description") or "").strip()
        tissue = (row.get("Tissue/Cell") or "").strip()
        up = (row.get("UniProt") or "").strip()
        if not up:
            continue
        up = up.split("-")[0].strip()
        if not up:
            continue
        if not condensate:
            continue
        if not desc or "N/A" in desc or desc == "Predicted from orthologs":
            continue
        if not tissue or "N/A" in tissue:
            continue
        uniprots.add(up)
    ordered = sorted(uniprots)
    with open(out_path, "wt", encoding="utf-8") as out:
        out.write("\n".join(ordered))
        if ordered:
            out.write("\n")

## Main Process

In [7]:
def positive_union(
    PhaSePro_path="phasepro_screened.txt",
    PhaSePDB_path="phasepdb_screened.txt",
    LLPSDB_path="llpsdb_screened.txt",
    DrLLPS_path="DrLLPS_screened.txt",
):
    with open(PhaSePro_path, "rt") as f:
        phasepro = set(f.readlines())
    with open(PhaSePDB_path, "rt") as f:
        phasepdb = set(f.readlines())
    with open(LLPSDB_path, "rt") as f:
        llpsdb = set(f.readlines())
    with open(DrLLPS_path, "rt") as f:
        drllps = set(f.readlines())
    positive = phasepro.union(phasepdb).union(llpsdb).union(drllps)
    with open("positive_all_taxon.txt", "wt") as f:
        f.writelines(positive)

get_phasepro()
get_llpsdb()
get_phasepdb()
drllps_first_screen()
drllps_second_screen()
positive_union()

# Length filter (50–3,000 amino acids)

In [ ]:
def filter_proteins_by_length(input_fasta, output_fasta, min_length=50, max_length=3000):
    """
    Filter proteins in a FASTA file by sequence length
    
    Args:
        input_fasta: Path to input FASTA file
        output_fasta: Path to output FASTA file
        min_length: Minimum sequence length (default 50)
        max_length: Maximum sequence length (default 3000)
    
    Returns:
        Tuple of (total_count, filtered_count) before and after filtering
    """
    total_count = 0
    filtered_count = 0
    filtered_records = []
    
    # Read and filter sequences
    for record in SeqIO.parse(input_fasta, "fasta"):
        total_count += 1
        seq_length = len(record.seq)
        
        if min_length <= seq_length <= max_length:
            filtered_count += 1
            filtered_records.append(record)
    
    # Write filtered sequences to output file
    if filtered_records:
        SeqIO.write(filtered_records, output_fasta, "fasta")
    
    return total_count, filtered_count

In [ ]:
input_file = "positive_all_taxon.fasta"  # Input FASTA
output_file = "positive_all_taxon_50-3000.fasta"  # Output FASTA

# Run filter
total, filtered = filter_proteins_by_length(input_file, output_file, min_length=50, max_length=3000)

# Print results
print(f"Proteins before filtering: {total}")
print(f"Proteins after filtering: {filtered}")
print(f"Proteins removed by filtering: {total - filtered}")
print(f"Fraction retained: {filtered/total*100:.2f}%")
print(f"\nFiltered sequences saved to: {output_file}")

# Exclude sequences with uncommon residues (e.g. B, J, O, U, X, Z)

In [ ]:
def find_non_standard_seqs(fasta_file: str) -> None:
    """List UniProt accessions whose sequence contains nonstandard amino acid letters."""
    # Set of 20 standard amino acids
    standard_aa = set("ACDEFGHIKLMNPQRSTVWY")
    
    non_standard_uids = []
    
    # Read FASTA
    for record in SeqIO.parse(fasta_file, "fasta"):
        # 1. Uppercase sequence (avoid case false positives)
        seq_str = str(record.seq).upper()
        seq_chars = set(seq_str)
        
        # 2. If sequence contains non-standard residues
        # (seq_chars - standard_aa) non-empty => non-standard
        if seq_chars - standard_aa:
            uid = extract_uniprot_id(record.description)
            if uid is None:
                uid = record.id
            non_standard_uids.append(uid)
            
    print(f"Done. Found {len(non_standard_uids)} sequences with non-standard amino acids.")
    
    return non_standard_uids


In [ ]:
remove_ids = find_non_standard_seqs('positive_all_taxon_50-3000.fasta')

In [ ]:
input_fasta = "positive_all_taxon_50-3000.fasta"
output_fasta = "positive_all_taxon_50-3000_removeNonstandard.fasta"

filter_fasta(input_fasta, remove_ids, output_fasta)

In [ ]:
input_fasta = "positive_all_taxon_50-3000_removeNonstandard.fasta"
output_file = "positive_all_taxon_50-3000_removeNonstandard.txt"

# Extract UniProt IDs
count = extract_uniprot_ids(input_fasta, output_file)

print(f"Successfully extracted {count} UniProt IDs")
print(f"UniProt IDs saved to: {output_file}")

# Taxonomic grouping

In [ ]:
def classify_species(taxonomic_lineage):
    """
    Classify organism from taxonomic lineage string
    
    Args:
        taxonomic_lineage (str): Taxonomic lineage string
    
    Returns:
        str: Class label
    """
    if not taxonomic_lineage or taxonomic_lineage == '':
        return 'Other'
    
    taxonomic_lineage = str(taxonomic_lineage).lower()
    
    # Check keywords in priority order
    if 'cellular organisms, bacteria' in taxonomic_lineage:
        return 'Bacteria'
    elif 'cellular organisms, archaea' in taxonomic_lineage:
        return 'Archaea'
    elif 'cellular organisms, eukaryota, opisthokonta, fungi' in taxonomic_lineage:
        return 'Fungi'
    elif 'amniota, mammalia' in taxonomic_lineage:
        return 'Mammals'
    elif 'cellular organisms, eukaryota, viridiplantae' in taxonomic_lineage:
        return 'Plants'
    elif 'cellular organisms, eukaryota, opisthokonta, metazoa' in taxonomic_lineage:
        return 'Non-mammalian animals'
    elif 'viruses' in taxonomic_lineage:
        return 'Viruses'
    else:
        return 'Other'
    
def process_row(row):
    taxonomic_lineage = str(row.get('Taxon', ''))
    species_class = classify_species(taxonomic_lineage)
    is_matched = taxonomic_lineage != ''
    return species_class, is_matched

UniProt ID mapping to get taxonomic lineage (local execution): positive_all_taxon_50-3000_removeNonstandard_idmapping.tsv

In [ ]:
all_data = pd.read_csv(f'positive_all_taxon_50-3000_removeNonstandard_idmapping.tsv',sep='\t')
all_data['Taxon'] = all_data['Taxonomic lineage'].str.replace(r' \(.*?\)', '', regex=True).str.strip()


In [ ]:
# Apply row-wise
results = all_data.apply(process_row, axis=1, result_type='expand')
all_data['species_class'] = results[0]
all_data['is_matched'] = results[1]

# Counts
class_counts = all_data['species_class'].value_counts().to_dict()
matched_count = np.sum(all_data['is_matched'])

# Print summary
print("\nClassification counts:")
for class_name, count in sorted(class_counts.items()):
    print(f"  {class_name}: {count} records")

print(f"\nRecords with empty lineage: {len(all_data) - matched_count}")
print(f"Records with lineage: {matched_count}")
print(f"Match rate: {(matched_count / len(all_data) * 100):.2f}%")

In [ ]:
cols = ['From','Organism (ID)','species_class']
save_data = all_data[cols].copy()
save_data.columns = ['uniprot', 'Organism_ID', 'species_class']

# Mapping rules dict
mapping_rules = {
    'Archaea':'Prokaryotes',
    'Bacteria':'Prokaryotes',
    'Other': 'Protists',
    'Non-mammalian animals': 'Animals(NM)',
    'Mammals':'Animals(M)',
}

save_data['organism'] = save_data['species_class'].replace(mapping_rules)
save_data = save_data[save_data['organism']!='Protists']
save_data.to_csv('positive_all_taxon_50-3000_removeNonstandard_organism.csv', index=False)


# IDP versus non-IDP labeling

Residue-level disorder and binding scores from AIUPred pipeline (local execution): positive_iupred_bindingScore.txt

In [ ]:
from __future__ import annotations
import matplotlib.pyplot as plt
import pandas as pd
import os
import sys
import re
import ast


_POSITIVE_FLAG = "1"
_NEGATIVE_FLAG = "0"


def dilate(states: str, max_length: int) -> str:
    """String dilation as in MobiDB-lite."""
    states = f"{_POSITIVE_FLAG * max_length}{states}{_POSITIVE_FLAG * max_length}"

    for level in range(1, max_length + 1):
        old = f"{_POSITIVE_FLAG * level}{_NEGATIVE_FLAG * level}{_POSITIVE_FLAG * level}"
        new = f"{_POSITIVE_FLAG * level}{_POSITIVE_FLAG * level}{_POSITIVE_FLAG * level}"
        for _ in range(level + 1):
            states = states.replace(old, new)

    return states[max_length:-max_length]


def erode(states: str, max_length: int) -> str:
    """String erosion as in MobiDB-lite."""
    states = f"{_NEGATIVE_FLAG * max_length}{states}{_NEGATIVE_FLAG * max_length}"

    for level in range(1, max_length + 1):
        old = f"{_NEGATIVE_FLAG * level}{_POSITIVE_FLAG * level}{_NEGATIVE_FLAG * level}"
        new = f"{_NEGATIVE_FLAG * level}{_NEGATIVE_FLAG * level}{_NEGATIVE_FLAG * level}"
        for _ in range(level + 1):
            states = states.replace(old, new)

    return states[max_length:-max_length]


def _repl_struct_by_disord(match: re.Match) -> str:
    """Replace matched long IDR + short gap + long IDR spans with '1'."""
    return _POSITIVE_FLAG * len(match.group(0))


def merge_long_disordered_regions(states: str) -> str:
    """Same as merge_long_disordered_regions in consensus.py."""
    pattern = r"{p}{{21,}}{n}{{1,10}}{p}{{21,}}".format(
        p=_POSITIVE_FLAG,
        n=_NEGATIVE_FLAG
    )

    while True:
        new_states = re.sub(
            pattern=pattern,
            repl=_repl_struct_by_disord,
            string=states
        )
        if new_states == states:
            return new_states
        states = new_states


def get_regions(states: str, min_length: int) -> list:
    """
    As get_regions in consensus.py:
    Return 0-based (start, end, flag) for flag != '0' and length >= min_length.
    """
    regions = []
    start = None
    current_flag = None

    for i, flag in enumerate(states):
        if flag != current_flag:
            if start is not None and current_flag != _NEGATIVE_FLAG:
                end = i - 1
                length = end - start + 1
                if length >= min_length:
                    regions.append((start, end, current_flag))
            start = i
            current_flag = flag

    if start is not None and current_flag != _NEGATIVE_FLAG:
        end = len(states) - 1
        length = end - start + 1
        if length >= min_length:
            regions.append((start, end, current_flag))

    return regions


def mobidb_lite_postproc_from_aiupred(scores, thr=0.5):
    """
    MobiDB-lite-style post-processing:
      1) score>=thr -> '1' / '0'
      2) dilate(max_length=3)
      3) erode(max_length=3)
      4) merge_long_disordered_regions
      5) get_regions(min_length=20), long IDRs only
    Returns: list of (start, end), inclusive 0-based
    """
    # 1) AIUPred scores -> initial state string
    states = "".join(
        _POSITIVE_FLAG if s >= thr else _NEGATIVE_FLAG
        for s in scores
    )

    # 2)–4) Morphology + gap merge (MobiDB-lite)
    states = dilate(states, max_length=3)
    states = erode(states, max_length=3)
    states = merge_long_disordered_regions(states)

    # 5) Extract long IDRs (>=20)
    regions = get_regions(states, min_length=20)
    return [(start, end) for (start, end, _) in regions]


def parse_aiupred_multi_fasta(path):
    proteins = []
    current_id = None
    current_scores = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            if line.startswith("# "):
                continue

            if line.startswith("#>"):
                if current_id is not None:
                    proteins.append({
                        "uniprot": current_id,
                        "scores": current_scores,
                    })
                parts = line.lstrip("#>").split("|")
                if len(parts) >= 2:
                    current_id = parts[1]
                else:
                    current_id = line.lstrip("#>").strip()
                current_scores = []
            else:
                cols = line.split()
                if len(cols) < 3:
                    continue
                score = float(cols[2])
                current_scores.append(score)

    if current_id is not None:
        proteins.append({
            "uniprot": current_id,
            "scores": current_scores,
        })

    return proteins


def aiupred_to_long_idr_df_mobidb_like(path, thr=0.5):
    """
    Read AIUPred output -> MobiDB-lite postprocess for long IDRs ->
    Returns DataFrame:
        uniprot | regions_0based
        P12345  | "[(53, 78), (317, 349)]"
    """
    proteins = parse_aiupred_multi_fasta(path)

    records = []
    for prot in proteins:
        uid = prot["uniprot"]
        scores = prot["scores"]
        regions = mobidb_lite_postproc_from_aiupred(scores, thr=thr)
        records.append({
            "uniprot": uid,
            "regions_0based": str(regions),
        })

    return pd.DataFrame(records)

In [ ]:
data = aiupred_to_long_idr_df_mobidb_like('positive_all_taxon_50-3000_removeNonstandard_iupred_bindingScore.txt')

data["regions_list"] = data["regions_0based"].apply(ast.literal_eval)
data["IDP_type"] = data["regions_list"].apply(
    lambda lst: "IDP" if len(lst) > 0 else "non-IDP"
)

data.to_csv('positive_all_taxon_50-3000_removeNonstandard_iupred_idr_regions.tsv', sep='\t', index=False)


# Define strata (taxonomy × IDP / non-IDP)

In [ ]:
IDR_df = pd.read_csv('positive_all_taxon_50-3000_removeNonstandard_iupred_idr_regions.tsv', sep='\t')
species_df = pd.read_csv('positive_all_taxon_50-3000_removeNonstandard_organism.csv')

class_df = pd.merge(species_df, IDR_df, on='uniprot')
class_df["class"] = class_df["organism"].astype(str) + "_" + class_df["IDP_type"].astype(str)
class_df.to_csv(
    'positive_class.tsv',
    sep='\t',
    index=False
)
